# Lista 4 - Kontekst korutyn, współdzielenie Flow i kanały

**Termin wykonania** – gr. wt. 15.12, gr. czw. 17.12

**Zadanie praktyczne: 6 pkt. Odpowiedź ustna: 4 pkt. Łącznie: 10 pkt.**

## Zadanie 1 - Dyspozytornia zgłoszeń (6 pkt)

Zaimplementuj w Kotlinie i Jetpack Compose aplikację, w której użytkownik dodaje zgłoszenia do kolejki i uruchamia odbiór kolejnego zgłoszenia. Jeśli kolejka jest pusta, dyspozytor czeka maksymalnie pięć sekund. Przyjęte do obsługi zgłoszenie trafia do historii. Można wyświetlać wszystkie pozycje historii albo tylko pilne.

Utwórz odrębny projekt. Dane i historia mogą być przechowywane w pamięci. Wystarczy jeden ekran, przykładowe dane i proste przyciski.

### 1. Model i podział odpowiedzialności - 1 pkt

- Zgłoszenie zawiera unikatowy identyfikator, krótką treść i kategorię „Standardowe” albo „Pilne”. Możesz generować treść i identyfikator automatycznie po naciśnięciu „Dodaj standardowe” lub „Dodaj pilne”. Kategoria służy do filtrowania historii. Kolejka obsługuje oba rodzaje w kolejności dodania.
- ViewModel zarządza kolejką, oczekiwaniem i historią. Ekran wyświetla stan i zgłasza akcje użytkownika. Pokaż wynik ostatniego dodania, status dyspozytora i historię z identyfikatorem, treścią oraz kategorią każdego odebranego zgłoszenia.
- Stan początkowy obejmuje pustą kolejkę, pustą historię i brak trwającego oczekiwania.

### 2. Kolejka i wynik trySend - 1 pkt

- Użyj jednego `Channel<Zgloszenie>(capacity = 2)` z domyślną strategią `BufferOverflow.SUSPEND`. Wysyłaj zgłoszenia przez `trySend()`. Kolejka przechowuje najwyżej dwa zgłoszenia oczekujące na odbiór.
- Odczytaj `ChannelResult`. Rozróżnij przyjęcie zgłoszenia do kolejki, zapełnienie kolejki i zamknięcie kanału. Pokaż odpowiedni komunikat. Zgłoszenie odrzucone przez kanał nie może pojawić się później w historii.
- Przy pustej historii dodaj trzy zgłoszenia bez uruchamiania odbioru. Pierwsze dwa powinny zostać przyjęte, a przy trzecim ma pojawić się komunikat „Kolejka pełna”.

### 3. Odbiór zgłoszenia lub upływ czasu - 1 pkt

- Przycisk „Odbierz następne” rozpoczyna jedną próbę odbioru w `viewModelScope`. W danej chwili może trwać najwyżej jedna próba. Przyciski dodawania mają działać także podczas oczekiwania.
- W jednej instrukcji `select` zaczekaj na zgłoszenie z kanału albo na `onTimeout(5_000)`. Każde kliknięcie po zakończonej próbie rozpoczyna własne pięć sekund oczekiwania. Gdy zgłoszenie już czeka w kolejce, odbierz je bez czekania na upływ limitu.
- Po odbiorze dopisz dokładnie jedną pozycję do historii i pokaż identyfikator odebranego zgłoszenia. Przy przekroczeniu czasu historia się nie zmienia. Po każdym wyniku można rozpocząć następną próbę.

Skorzystaj z dokumentacji [`onTimeout`](https://kotlinlang.org/api/kotlinx.coroutines/kotlinx-coroutines-core/kotlinx.coroutines.selects/on-timeout.html).

### 4. Zamknięcie kanału i opróżnianie kolejki - 1 pkt

- Przycisk „Zamknij kolejkę” wywołuje `close()`. Po zamknięciu kolejne próby dodania mają dać komunikat „Kolejka zamknięta”. Przyjęte wcześniej zgłoszenia nadal odbieraj po jednym, kolejnymi kliknięciami „Odbierz następne”.
- Gdy zamknięty kanał zostanie opróżniony, próba odbioru od razu pokazuje komunikat „Kolejka zamknięta i pusta”. Jeżeli użytkownik zamknie pustą kolejkę podczas oczekiwania, odbiór także ma zakończyć się bez czekania na upływ limitu.
- Rozróżnij te wyniki w gałęziach `select`. [`onReceiveCatching`](https://kotlinlang.org/api/kotlinx.coroutines/kotlinx-coroutines-core/kotlinx.coroutines.channels/-receive-channel/on-receive-catching.html) daje `ChannelResult` z odebranym elementem albo informacją o zamknięciu. Ponowne otwieranie kolejki w tej instancji ekranu nie jest wymagane.

### 5. Historia i filtr przez combine - 1 pkt

- Przechowuj pełną historię i wybór filtra w dwóch oddzielnych źródłach, np. w prywatnych `MutableStateFlow`. Dodaj filtr „Wszystkie” / „Tylko pilne”.
- Użyj `combine`, aby wyznaczyć widoczne pozycje na podstawie aktualnej historii i filtra. Zmiana dowolnego z tych wejść ma odświeżyć widok. Pełna historia pozostaje dostępna po powrocie do filtra „Wszystkie”.
- Filtr dotyczy historii. Nie zmienia kolejności odbioru z kanału ani zawartości kolejki.

### 6. StateFlow i cykl życia ekranu - 1 pkt

- W ViewModelu użyj `stateIn` z `viewModelScope`, `SharingStarted.WhileSubscribed(5_000)` i sensowną wartością początkową. Udostępnij `StateFlow` ze stanem ekranu zawierającym wyniki akcji, filtr i widoczną historię. W Compose użyj `collectAsStateWithLifecycle()`.
- Uruchamiaj obsługę kolejki w reakcji na akcje użytkownika. Zmiana filtra, rekompozycja ani ponowne dołączenie kolektora UI nie mogą uruchomić dodatkowego odbioru, odtworzyć zgłoszeń ani utworzyć nowej kolejki w tym samym ViewModelu.
- Aktywna próba odbioru ma własny `Job` w `viewModelScope`. Krótkie przejście do tła lub obrót ekranu nie powinny zerować jej limitu. Gdy UI wróci, wyświetla stan tej samej próby albo jej wynik. Zapis po zakończeniu procesu nie jest wymagany.

### Punktacja zadania i prezentacja rozwiązania

Każda z sześciu części jest oceniana osobno, maksymalnie na **1 pkt**. Do kodu mogą pojawić się dodatkowe pytania sprawdzające zrozumienie i samodzielne wykonanie implementacji.

Na laboratorium uruchom aplikację na emulatorze lub urządzeniu. Pokaż działanie aplikacji i wskaż w kodzie miejsca, w których używasz `Channel`, `select`, `combine` i `stateIn`.

| Scenariusz | Oczekiwany wynik |
|:---|:---|
| Dodaj trzy zgłoszenia bez odbioru. | Dwa zostają przyjęte, trzecie jest odrzucone. Dwa kolejne odbiory zwracają przyjęte zgłoszenia w kolejności dodania. |
| Zacznij odbiór z pustego, otwartego kanału i dodaj zgłoszenie przed końcem limitu. | Oczekiwanie kończy się od razu po dodaniu, a zgłoszenie pojawia się raz w historii. |
| Od razu zacznij kolejną próbę z pustego, otwartego kanału i nic nie dodawaj. | Próba kończy się po własnych około pięciu sekundach, a historia się nie zmienia. |
| Odbierz co najmniej jedno zgłoszenie z każdej kategorii. Przełącz filtr i odbierz kolejne pilne zgłoszenie. | Przy filtrze widać tylko pilne, także nowo odebrane. Po przywróceniu „Wszystkie” wraca pełna historia. |
| Rozpocznij odbiór z pustej kolejki i przejdź na krótko do tła albo obróć urządzenie. | Ta sama próba kończy się w swoim czasie. Po powrocie widać jej wynik albo trwające oczekiwanie. |
| Dodaj zgłoszenie i zamknij kanał. Spróbuj dodać następne, po czym dwukrotnie naciśnij „Odbierz następne”. | Dodanie zostaje odrzucone. Pierwszy odbiór zwraca oczekujące zgłoszenie, a drugi od razu informuje o zamknięciu i braku elementów. |

## Odpowiedź ustna - 4 pkt

- Student losuje **4 różne pytania** z całej puli **1–22**. Podział tematyczny służy porządkowaniu materiału.
- Za każde pytanie można otrzymać **0 albo 1 pkt**. Liczba losowanych pytań jest stała i nie zależy od punktów za aplikację.
- Uzasadnienie lub przewidywany wynik jest częścią odpowiedzi, jeśli wymaga go treść pytania.

## Pula pytań do losowania

Pytania obejmują wykłady 6 i 7 PUM2. Każde jest warte **1 pkt**.

### A. Kontekst wykonania i flowOn

1. Co zwraca `withContext` i w jakim kontekście wykona się dalszy kod po zakończeniu jego bloku? **(1 pkt)**
2. Który dispatcher wybierzesz do intensywnych obliczeń, a który do blokującego odczytu pliku? Uzasadnij wybór. **(1 pkt)**
3. Czym różni się oczekiwanie na wynik `withContext(Dispatchers.Default)` od uruchomienia `launch(Dispatchers.Default)`? **(1 pkt)**
4. Załóż, że `collect` działa na Main. W łańcuchu `flow { … }.flowOn(Dispatchers.IO).map { … }.collect { … }` wskaż kontekst bloku `flow`, `map` i `collect`. **(1 pkt)**
5. Gdzie w łańcuchu `flow { … }.map { … }.collect { … }` wstawić `flowOn`, aby jego dispatcher objął również `map`? **(1 pkt)**

### B. Współdzielenie strumieni i stan ekranu

6. Za co odpowiadają `scope`, `started` i `initialValue` w wariancie `stateIn` używanym w ViewModelu? **(1 pkt)**
8. Kiedy pierwszy raz zaczynają zbierać źródło `Eagerly`, `Lazily` i `WhileSubscribed`? **(1 pkt)**
9. W `WhileSubscribed(5_000)` ostatni odbiorca znika na 3 sekundy. Co dzieje się z nadal działającym źródłem, gdy odbiorca wraca? **(1 pkt)**
10. Ostatni odbiorca znika na 8 sekund przy `WhileSubscribed(5_000)`. Co dzieje się ze źródłem i ostatnią wartością `StateFlow` przy domyślnym czasie wygaśnięcia cache? **(1 pkt)**
11. Co zmieni `replayExpirationMillis = 0` w `stateIn`, gdy `WhileSubscribed` zatrzyma zbieranie źródła? **(1 pkt)**
12. Czym różni się `asStateFlow()` wywołane na `MutableStateFlow` od `stateIn(...)` wywołanego na istniejącym `Flow`? **(1 pkt)**
13. Jak to możliwe, że po restarcie zimnego źródła `shareIn(replay = 0)` odbiorca widzi komunikat o tej samej treści co przed zatrzymaniem? **(1 pkt)**
14. Jedno ze źródeł `combine` już nadało wartość, drugie jeszcze nie. Czy `combine` może wtedy zwrócić pierwszy wynik? Uzasadnij. **(1 pkt)**

### C. Kanały, bufor i zamykanie

14. Kanał o pojemności 2 z domyślną strategią ma dwa elementy. Co zrobi trzecie `send()`, jeśli nikt jeszcze nie odbiera? **(1 pkt)**
19. Producent wysyła `0`, `1`, `2` do kanału o pojemności 2, a odbiór zaczyna się później. Jakie elementy zostają przy `DROP_OLDEST`, a jakie przy `DROP_LATEST`? **(1 pkt)**
20. Producent wysłał `0`, `1`, `2` do `Channel.CONFLATED` przed pierwszym odbiorem. Co odbierze konsument i dlaczego? **(1 pkt)**
21. Co zwraca `trySend()`? Jak odróżnisz pełny kanał z `SUSPEND` od kanału zamkniętego? **(1 pkt)**
22. Czym różni się `tryReceive()` od `receive()` na pustym, otwartym kanale? **(1 pkt)**
23. W buforze są dwa elementy. Co stanie się z nimi po `close()`, a co po `cancel()`? **(1 pkt)**

### D. select i czas życia operacji

20. Na co czeka `select` i jaka wartość staje się wynikiem tego wyrażenia? **(1 pkt)**
28. Dwa kanały mają element gotowy do odbioru. Od czego zależy wybór gałęzi w zwykłym `select`? **(1 pkt)**
29. Co różni pięć sekund w `onTimeout(5_000)` od pięciu sekund w `SharingStarted.WhileSubscribed(5_000)`? Wskaż, które oczekiwanie dotyczy odbioru z kanału, a które zbierania źródła Flow. **(1 pkt)**

## Oceny

Punkty za zadanie i odpowiedzi sumują się.

| Punkty łącznie | Ocena |
|:---|:---:|
| poniżej 6 | 2,0 |
| 6 | 3,0 |
| 7 | 3,5 |
| 8 | 4,0 |
| 9 | 4,5 |
| 10 | 5,0 |